In [0]:
# ==========================================================
# NOTEBOOK : 04_SCD_PROCESS
# PURPOSE  : Apply SCD Type 2 on Customer and Product
# ==========================================================

from pyspark.sql import functions as F

print("SCD Type 2 Notebook Started")

# ==========================================================
# STORAGE CONFIGURATION
# ==========================================================

storage_account = "retailadls67"
access_key = "bhy6cfJKxIz/zKE9cm/ofV8FACSlK06rqmC28MJPQSuWbFvYRiBKNs3j4s0pne0qo/erb+PxN+zx+AStajlJDg=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

# ==========================================================
# SILVER INPUT PATHS
# ==========================================================

SILVER_CUSTOMERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/customers"
SILVER_PRODUCTS = f"abfss://silver@{storage_account}.dfs.core.windows.net/products"

# ==========================================================
# GOLD OUTPUT PATHS
# ==========================================================

GOLD_DIM_CUSTOMER = f"abfss://gold@{storage_account}.dfs.core.windows.net/dim_customer_scd2"
GOLD_DIM_PRODUCT = f"abfss://gold@{storage_account}.dfs.core.windows.net/dim_product_scd2"

# ==========================================================
# READ SILVER DELTA
# ==========================================================

customer_df = spark.read.format("delta").load(SILVER_CUSTOMERS)
product_df = spark.read.format("delta").load(SILVER_PRODUCTS)

# ==========================================================
# CUSTOMER SCD TYPE 2
# ==========================================================

customer_df = customer_df.withColumn(
    "CustomerSegment",
    F.when(
        (F.col("CustomerID") % 10) == 0,
        "Premium"
    ).otherwise(F.col("CustomerSegment"))
)

customer_scd2 = (
    customer_df
    .withColumn("EffectiveDate", F.current_date())
    .withColumn("ExpiryDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit("Y"))
    .withColumn("Version", F.lit(1))
)

# ==========================================================
# PRODUCT SCD TYPE 2
# ==========================================================

product_df = product_df.withColumn(
    "SellingPrice",
    F.when(
        (F.col("ProductID") % 10) == 0,
        F.col("SellingPrice") + 500
    ).otherwise(F.col("SellingPrice"))
)

product_df = product_df.withColumn(
    "Status",
    F.when(
        (F.col("ProductID") % 20) == 0,
        "Discontinued"
    ).otherwise(F.col("Status"))
)

product_scd2 = (
    product_df
    .withColumn("EffectiveDate", F.current_date())
    .withColumn("ExpiryDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit("Y"))
    .withColumn("Version", F.lit(1))
)

# ==========================================================
# WRITE GOLD DELTA
# ==========================================================

customer_scd2.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_DIM_CUSTOMER)

product_scd2.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_DIM_PRODUCT)

# ==========================================================
# VALIDATION
# ==========================================================

print("=" * 50)
print("SCD TYPE 2 COMPLETED")
print("=" * 50)

print("Customer Records :", customer_scd2.count())
print("Product Records  :", product_scd2.count())

customer_scd2.display(5)
product_scd2.display(5)